# Frozen external RevalExo evaluation

Train the selected pooled Inception model on the complete internal development set, then evaluate RevalExo once as an external cohort. RevalExo is not used for normalization, calibration, threshold selection, or model fitting. The model input is the established 3-channel acceleration-magnitude representation.

In [1]:
import random
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
from torch import nn
from torch.utils.data import Dataset, DataLoader
PROJECT_ROOT=Path.cwd().resolve(); PROJECT_ROOT=PROJECT_ROOT.parent if PROJECT_ROOT.name.lower()=='notebooks' else PROJECT_ROOT
P=PROJECT_ROOT/'data'/'processed'; DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X=np.load(P/'validated_acceleration_magnitude_windows_float32.npy',mmap_mode='r'); md=pd.read_csv(P/'validated_window_metadata.csv'); md['label_binary']=md['label'].map({'healthy':0,'stroke':1}).astype(int)
E=np.load(P/'revalexo_external_windows_float32.npy',mmap_mode='r'); em=pd.read_csv(P/'revalexo_external_window_metadata.csv'); em['label_binary']=em['group'].map({'HC':0,'ST':1}).astype(int)
# 18-channel RevalExo order: LB acc/gyro, RF acc/gyro, LF acc/gyro. Derive the same 3 acceleration magnitudes used internally.
E_mag=np.stack([np.linalg.norm(E[:,:,0:3],axis=2),np.linalg.norm(E[:,:,6:9],axis=2),np.linalg.norm(E[:,:,12:15],axis=2)],axis=2).astype('float32')
print('Device:',DEVICE,'internal:',X.shape,'external:',E_mag.shape)

Device: cuda internal: (18511, 500, 3) external: (2228, 500, 3)


In [2]:
def seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s) if torch.cuda.is_available() else None
total=np.zeros(3); sq=np.zeros(3); n=0
for i in range(0,len(X),512):
 b=np.asarray(X[i:i+512],dtype='float32'); total+=b.sum((0,1)); sq+=(b*b).sum((0,1)); n+=b.shape[0]*b.shape[1]
mean=(total/n).astype('float32'); std=np.sqrt(np.maximum(sq/n-mean**2,1e-8)).astype('float32')
counts=md.groupby('participant_key').size(); cells=md.groupby(['dataset_id','label_binary']).participant_key.nunique(); weights=(md.participant_key.map(1/counts).to_numpy()*np.array([1/cells[(r.dataset_id,r.label_binary)] for r in md.itertuples()])); weights=(weights/weights.mean()).astype('float32')
class DS(Dataset):
 def __init__(self,arr,frame,weights=None): self.arr=arr; self.frame=frame; self.weights=weights
 def __len__(self): return len(self.frame)
 def __getitem__(self,i):
  j=int(self.frame.index[i]); z=((np.asarray(self.arr[j],dtype='float32')-mean)/std).T.copy(); w=1 if self.weights is None else self.weights[i]
  return torch.from_numpy(z),torch.tensor(float(self.frame.iloc[i].label_binary)),torch.tensor(float(w))
class Block(nn.Module):
 def __init__(self,c,o=16):
  super().__init__(); b=min(32,c); self.b=nn.Conv1d(c,b,1,bias=False); self.br=nn.ModuleList([nn.Conv1d(b,o,k,padding=k//2,bias=False) for k in (7,15,25)]); self.p=nn.Conv1d(c,o,1,bias=False); self.bn=nn.BatchNorm1d(o*4); self.r=nn.Conv1d(c,o*4,1,bias=False) if c!=o*4 else nn.Identity()
 def forward(self,x):
  z=self.b(x); q=[v(z) for v in self.br]+[self.p(nn.functional.max_pool1d(x,3,1,1))]; return nn.functional.gelu(self.bn(torch.cat(q,1))+self.r(x))
class Net(nn.Module):
 def __init__(self): super().__init__(); self.f=nn.Sequential(Block(3),nn.MaxPool1d(2),Block(64),nn.AdaptiveAvgPool1d(1)); self.c=nn.Sequential(nn.Flatten(),nn.Dropout(.3),nn.Linear(64,1))
 def forward(self,x): return self.c(self.f(x)).squeeze(1)
def predict(model,loader):
 model.eval(); out=[]
 with torch.no_grad():
  for z,_,_ in loader: out.extend(model(z.to(DEVICE)).cpu().numpy())
 return np.asarray(out)
train_loader=DataLoader(DS(X,md,weights),batch_size=256,shuffle=True,num_workers=0)
ext_loader=DataLoader(DS(E_mag,em),batch_size=256,shuffle=False,num_workers=0)

In [3]:
all_ext=[]
for s in (42,52,62):
 seed(s); model=Net().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
 for epoch in range(6):
  model.train()
  for z,y,w in train_loader:
   opt.zero_grad(); loss=(nn.functional.binary_cross_entropy_with_logits(model(z.to(DEVICE)),y.to(DEVICE),reduction='none')*w.to(DEVICE)).mean(); loss.backward(); opt.step()
 logits=predict(model,ext_loader); all_ext.append(logits); print('seed',s,'epoch',epoch+1)
logits=np.mean(all_ext,axis=0); prob=1/(1+np.exp(-logits)); em['raw_probability']=prob
part=em.groupby(['subject','group','label_binary'],as_index=False).raw_probability.mean()
auc=roc_auc_score(part.label_binary,part.raw_probability); brier=brier_score_loss(part.label_binary,part.raw_probability)
print('Participant-level external AUROC:',round(auc,3),'Brier:',round(brier,3)); print(part.groupby('group').raw_probability.agg(['count','mean','median']).to_string())
em.to_csv(P/'revalexo_external_window_predictions.csv',index=False); part.to_csv(P/'revalexo_external_participant_predictions.csv',index=False)
pd.DataFrame([{'scope':'participant','participants':len(part),'auroc':auc,'brier':brier}]).to_csv(P/'revalexo_external_metrics.csv',index=False)

seed 42 epoch 6


seed 52 epoch 6


seed 62 epoch 6
Participant-level external AUROC: 0.871 Brier: 0.17
       count      mean    median
group                           
HC         7  0.541095  0.568352
ST        10  0.764305  0.721010
